# Colab Local LLM Classifier
Runs the classifier stage with a local Hugging Face Transformers model and writes classifier-only prediction parquet files for every configured main split and external target.


In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/noamdwc/llm-gatekeeping.git'
BRANCH = 'zero_shot_nlp_attack'
LIMIT = None  # Set to a small integer for a smoke test; None runs every row in every target.
MAIN_SPLITS = ['train', 'val', 'test', 'unseen_val', 'unseen_test', 'safeguard_test']
EXTERNAL_DATASETS = ['deepset', 'jackhhao']
MODEL_ID = 'meta/llama-3.1-8b-instruct'
HF_MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
MODEL_PROVIDER_NAME = 'transformers-local'
MAX_MODEL_LEN = 4096
BATCH_SIZE = 1
CHECKPOINT_EVERY = 50
OUTPUT_SUFFIX = 'colab_local_classifier'
LOAD_IN_4BIT = False
LOCAL_CAPTURE_LOGPROBS = True
LOCAL_TOP_LOGPROBS = 5

try:
    from google.colab import drive

    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/data/llm-gatekeeping'
    REPO_DIR = '/content/llm-gatekeeping'
    SKIP_REPO_SYNC = False
except ImportError:
    DRIVE_ROOT = os.environ.get('LLM_GATEKEEPING_DATA_ROOT', str(Path.cwd()))
    REPO_DIR = os.environ.get('LLM_GATEKEEPING_REPO_DIR', str(Path.cwd()))
    SKIP_REPO_SYNC = True

HF_CACHE_DIR = f'{DRIVE_ROOT}/cache/huggingface'
SPLITS_DIR = f'{DRIVE_ROOT}/data/processed/splits'
PREDICTIONS_DIR = f'{DRIVE_ROOT}/data/processed/predictions'
PREDICTIONS_EXTERNAL_DIR = f'{DRIVE_ROOT}/data/processed/predictions_external'

os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_HUB_CACHE'] = f'{HF_CACHE_DIR}/hub'
os.environ['TRANSFORMERS_CACHE'] = f'{HF_CACHE_DIR}/transformers'
os.environ['HF_DATASETS_CACHE'] = f'{HF_CACHE_DIR}/datasets'
print(f'Model ID: {MODEL_ID}')
print(f'Hugging Face model: {HF_MODEL_ID}')
print(f'Hugging Face cache: {HF_CACHE_DIR}')
print(f'Main output dir: {PREDICTIONS_DIR}')
print(f'External output dir: {PREDICTIONS_EXTERNAL_DIR}')
print(f'Targets: {len(MAIN_SPLITS)} main splits + {len(EXTERNAL_DATASETS)} external datasets')


In [ ]:
import os
import subprocess

if SKIP_REPO_SYNC:
    os.chdir(REPO_DIR)
    print(f'Using local repo at {os.getcwd()}')
else:
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        print(f'Using existing repo at {REPO_DIR}')

    os.chdir(REPO_DIR)
    print('Repo:', os.getcwd())

    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)


In [ ]:
import subprocess
import sys

packages = [
    'accelerate',
    'bitsandbytes',
    'datasets',
    'pyarrow',
    'sentencepiece',
    'transformers>=4.44.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


In [ ]:
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

Path(PREDICTIONS_DIR).mkdir(parents=True, exist_ok=True)
Path(PREDICTIONS_EXTERNAL_DIR).mkdir(parents=True, exist_ok=True)
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Enable a Colab GPU runtime before loading the local LLM.')

torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization_config = None
if LOAD_IN_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch_dtype,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
    )

tokenizer = AutoTokenizer.from_pretrained(
    HF_MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    trust_remote_code=True,
)
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

device_map = 'auto'
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    torch_dtype=torch_dtype,
    device_map=device_map,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
model.eval()
print(f'Loaded {HF_MODEL_ID} with dtype={torch_dtype}, device_map={device_map}, 4bit={LOAD_IN_4BIT}')


In [ ]:
print(f'Local Transformers model is ready: provider={MODEL_PROVIDER_NAME}, model={MODEL_ID}')
print(f'Pad token id: {tokenizer.pad_token_id}; EOS token id: {tokenizer.eos_token_id}')


In [ ]:
from pathlib import Path

import pandas as pd

from src.external_datasets import load_external_dataset
from src.llm_classifier.constants import NLP_TYPES
from src.llm_classifier.llm_classifier import HierarchicalLLMClassifier
from src.llm_classifier.llm_classifier import build_few_shot_examples
from src.llm_classifier.prompts import build_classifier_messages
from src.utils import build_sample_id, load_config

cfg = load_config()
text_col = cfg['dataset']['text_col']
label_col = cfg['dataset']['label_col']

train_path = Path(SPLITS_DIR) / 'train.parquet'
if not train_path.exists():
    raise FileNotFoundError(f'Missing train split: {train_path}')
train_df = pd.read_parquet(train_path)
missing_train = {text_col, label_col} - set(train_df.columns)
if missing_train:
    raise ValueError(f'{train_path} missing required columns: {sorted(missing_train)}')

few_shot, used_ids = build_few_shot_examples(train_df, cfg)
print(f'Train rows: {len(train_df):,}; few-shot pairs: {len(few_shot)}')


def make_main_target(split: str) -> dict:
    return {
        'kind': 'main',
        'name': split,
        'progress_label': f'main split {split}',
        'input_path': Path(SPLITS_DIR) / f'{split}.parquet',
        'checkpoint_path': Path(PREDICTIONS_DIR) / f'llm_checkpoint_{split}_{OUTPUT_SUFFIX}.parquet',
        'output_path': Path(PREDICTIONS_DIR) / f'llm_predictions_{split}_{OUTPUT_SUFFIX}.parquet',
    }


def make_external_target(dataset_key: str) -> dict:
    return {
        'kind': 'external',
        'name': dataset_key,
        'dataset_key': dataset_key,
        'progress_label': f'external dataset {dataset_key}',
        'checkpoint_path': Path(PREDICTIONS_EXTERNAL_DIR) / f'llm_checkpoint_external_{dataset_key}_{OUTPUT_SUFFIX}.parquet',
        'output_path': Path(PREDICTIONS_EXTERNAL_DIR) / f'llm_predictions_external_{dataset_key}.parquet',
    }


def load_main_target_df(target: dict) -> pd.DataFrame:
    path = target['input_path']
    if not path.exists():
        raise FileNotFoundError(f"Missing {target['progress_label']}: {path}")
    df = pd.read_parquet(path)
    missing = {text_col} - set(df.columns)
    if missing:
        raise ValueError(f'{path} missing required columns: {sorted(missing)}')
    return df


def load_external_target_df(target: dict) -> pd.DataFrame:
    dataset_key = target['dataset_key']
    ds_cfg = cfg['external_datasets'][dataset_key]
    df = load_external_dataset(ds_cfg)
    missing = {text_col} - set(df.columns)
    if missing:
        raise ValueError(f"{target['progress_label']} missing required columns: {sorted(missing)}")
    return df


def prepare_target_df(target: dict) -> pd.DataFrame:
    if target['kind'] == 'main':
        df = load_main_target_df(target)
    elif target['kind'] == 'external':
        df = load_external_target_df(target)
    else:
        raise ValueError(f"Unknown target kind: {target['kind']}")
    if LIMIT is not None:
        df = df.head(int(LIMIT)).copy()
    else:
        df = df.copy()
    df['sample_id'] = df[text_col].apply(build_sample_id)
    df = df.drop_duplicates(subset=['sample_id'], keep='last').reset_index(drop=True)
    print(f"Prepared {target['progress_label']}: {len(df):,} rows")
    return df


In [ ]:
import json

VALID_NLP_TYPES = set(NLP_TYPES) | {'none'}


def build_static_few_shot_messages(text: str) -> list[dict]:
    messages = []
    for benign_text, attack_text, attack_type in few_shot:
        messages.append({'role': 'user', 'content': f'INPUT_PROMPT:\n{benign_text}'})
        messages.append({
            'role': 'assistant',
            'content': json.dumps({
                'label': 'benign',
                'confidence': 95,
                'nlp_attack_type': 'none',
                'evidence': '',
                'reason': 'No active attempt to override instructions, exfiltrate data, or hijack tools.',
            }),
        })
        if attack_type in NLP_TYPES:
            evidence = ''
            adv_reason = f'Perturbed tokens characteristic of {attack_type} adversarial attack.'
        else:
            evidence = attack_text[:80]
            adv_reason = f'Contains {attack_type} obfuscation; active adversarial prompt detected.'
        messages.append({'role': 'user', 'content': f'INPUT_PROMPT:\n{attack_text}'})
        messages.append({
            'role': 'assistant',
            'content': json.dumps({
                'label': 'adversarial',
                'confidence': 84,
                'nlp_attack_type': attack_type if attack_type in NLP_TYPES else 'none',
                'evidence': evidence,
                'reason': adv_reason,
            }),
        })
    return messages


def format_chat_prompt(messages: list[dict]) -> str:
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    rendered = []
    for message in messages:
        role = message.get('role', 'user').upper()
        rendered.append(f'{role}: {message.get("content", "")}')
    rendered.append('ASSISTANT:')
    return '\n\n'.join(rendered)


def extract_generated_token_logprobs(generated_ids, scores, row_idx: int = 0) -> list[dict] | None:
    if not LOCAL_CAPTURE_LOGPROBS or scores is None:
        return None
    entries = []
    for token_id, score in zip(generated_ids.tolist(), scores):
        log_probs = torch.log_softmax(score[row_idx].detach().float().cpu(), dim=-1)
        token_logprob = float(log_probs[token_id].item())
        top_k = min(int(LOCAL_TOP_LOGPROBS), int(log_probs.numel()))
        top_values, top_indices = torch.topk(log_probs, k=top_k)
        top_logprobs = [
            {
                'token': tokenizer.decode([int(top_token_id)], skip_special_tokens=False),
                'token_id': int(top_token_id),
                'logprob': float(top_logprob),
            }
            for top_logprob, top_token_id in zip(top_values.tolist(), top_indices.tolist())
        ]
        entries.append({
            'token': tokenizer.decode([token_id], skip_special_tokens=False),
            'token_id': int(token_id),
            'logprob': token_logprob,
            'top_logprobs': top_logprobs,
        })
    return entries


def normalize_classifier_payload(payload: dict, raw_response_text: str | None, token_logprobs: list[dict] | None) -> dict:
    if not isinstance(payload, dict) or not payload:
        raw_response_text = raw_response_text or ''
        label = 'adversarial'
        nlp_attack_type = 'none'
        category = HierarchicalLLMClassifier._derive_category(label, nlp_attack_type)
        return {
            'label': label,
            'confidence': 0.0,
            'nlp_attack_type': nlp_attack_type,
            'evidence': 'parse_failure',
            'reason': 'classifier response could not be parsed',
            '_token_logprobs': token_logprobs,
            '_provider_name': MODEL_PROVIDER_NAME,
            '_model_name': MODEL_ID,
            '_raw_response_text': raw_response_text,
            '_parse_success': False,
            '_category': category,
            'clf_label': label,
            'clf_category': category,
            'clf_confidence': 0.0,
            'clf_evidence': 'parse_failure',
            'clf_nlp_attack_type': nlp_attack_type,
            'clf_provider_name': MODEL_PROVIDER_NAME,
            'clf_model_name': MODEL_ID,
            'clf_raw_response_text': raw_response_text,
            'clf_parse_success': False,
            'clf_token_logprobs': token_logprobs,
        }

    label = payload.get('label', '') if payload else ''
    if label not in ('benign', 'adversarial', 'uncertain'):
        label = 'adversarial'

    nlp_attack_type = payload.get('nlp_attack_type', 'none') if payload else 'none'
    if nlp_attack_type not in VALID_NLP_TYPES:
        nlp_attack_type = 'none'
    if label != 'adversarial':
        nlp_attack_type = 'none'

    evidence = payload.get('evidence', '') if payload else ''
    if label != 'adversarial':
        evidence = ''

    confidence = HierarchicalLLMClassifier._normalize_confidence(
        payload.get('confidence', 50) if payload else 50
    )
    category = HierarchicalLLMClassifier._derive_category(label, nlp_attack_type)

    return {
        'label': label,
        'confidence': confidence,
        'nlp_attack_type': nlp_attack_type,
        'evidence': evidence,
        'reason': payload.get('reason', '') if payload else '',
        '_token_logprobs': token_logprobs,
        '_provider_name': MODEL_PROVIDER_NAME,
        '_model_name': MODEL_ID,
        '_raw_response_text': raw_response_text,
        '_parse_success': bool(payload),
        '_category': category,
        'clf_label': label,
        'clf_category': category,
        'clf_confidence': confidence,
        'clf_evidence': evidence,
        'clf_nlp_attack_type': nlp_attack_type,
        'clf_provider_name': MODEL_PROVIDER_NAME,
        'clf_model_name': MODEL_ID,
        'clf_raw_response_text': raw_response_text,
        'clf_parse_success': bool(payload),
        'clf_token_logprobs': token_logprobs,
    }


def parse_classifier_json(raw_response_text: str | None) -> dict:
    if raw_response_text is None:
        return {}
    text = raw_response_text.strip()
    if '```' in text:
        text = text.replace('```json', '```')
        parts = text.split('```')
        if len(parts) >= 3:
            text = parts[1].strip()
    try:
        payload = json.loads(text)
        if isinstance(payload, str):
            payload = json.loads(payload)
        return payload if isinstance(payload, dict) else {}
    except json.JSONDecodeError:
        start = text.find('{')
        end = text.rfind('}')
        if start >= 0 and end > start:
            try:
                payload = json.loads(text[start:end + 1])
                return payload if isinstance(payload, dict) else {}
            except json.JSONDecodeError:
                return {}
        return {}


def classify_texts(texts: list[str]) -> list[dict]:
    if not texts:
        return []
    prompts = [format_chat_prompt(build_classifier_messages(text, build_static_few_shot_messages(text))) for text in texts]
    model_inputs = tokenizer(
        prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_MODEL_LEN,
    ).to(model.device)
    do_sample = cfg['llm']['temperature'] > 0
    generation_kwargs = {
        'max_new_tokens': cfg['llm']['max_tokens_classifier'],
        'do_sample': do_sample,
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'return_dict_in_generate': True,
        'output_scores': LOCAL_CAPTURE_LOGPROBS,
    }
    if do_sample:
        generation_kwargs['temperature'] = max(float(cfg['llm']['temperature']), 1e-6)
        generation_kwargs['top_p'] = float(cfg['llm'].get('top_p', 1.0))
    try:
        with torch.inference_mode():
            generated = model.generate(**model_inputs, **generation_kwargs)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise

    prompt_length = model_inputs['input_ids'].shape[1]
    results = []
    for row_idx, sequence in enumerate(generated.sequences):
        generated_ids = sequence[prompt_length:]
        raw_response_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        token_logprobs = extract_generated_token_logprobs(generated_ids, generated.scores if LOCAL_CAPTURE_LOGPROBS else None, row_idx=row_idx)
        payload = parse_classifier_json(raw_response_text)
        results.append(normalize_classifier_payload(payload, raw_response_text, token_logprobs))
    return results


def classify_text(text: str) -> dict:
    return classify_texts([text])[0]


In [ ]:
GROUND_TRUTH_COLUMNS = [
    'modified_sample',
    'original_sample',
    'attack_name',
    'label_binary',
    'label_category',
    'label_type',
    'prompt_hash',
    'benign_source',
    'is_synthetic_benign',
]
PREDICTION_COLUMNS = [
    'llm_pred_binary',
    'llm_pred_raw',
    'llm_pred_category',
    'llm_conf_binary',
    'llm_evidence',
    'llm_stages_run',
    'llm_provider_name',
    'llm_model_name',
    'llm_raw_response_text',
    'llm_parse_success',
    'clf_label',
    'clf_category',
    'clf_confidence',
    'clf_evidence',
    'clf_nlp_attack_type',
    'clf_provider_name',
    'clf_model_name',
    'clf_raw_response_text',
    'clf_parse_success',
    'clf_token_logprobs',
]


def valid_prediction_mask(df: pd.DataFrame) -> pd.Series:
    required_columns = {'sample_id', *PREDICTION_COLUMNS}
    if not required_columns.issubset(df.columns):
        return pd.Series(False, index=df.index)
    required_non_null_columns = ['sample_id', *PREDICTION_COLUMNS]
    return (
        df[required_non_null_columns].notna().all(axis=1)
        & df['llm_stages_run'].eq(1)
        & df['llm_pred_binary'].isin({'benign', 'adversarial'})
        & df['llm_pred_raw'].isin({'benign', 'adversarial', 'uncertain'})
        & df['llm_conf_binary'].notna()
        & df['clf_confidence'].notna()
        & df['llm_parse_success'].notna()
    )


def assert_valid_output(out_df: pd.DataFrame, target: dict) -> None:
    expected_columns = {'sample_id', *PREDICTION_COLUMNS}
    missing_columns = expected_columns - set(out_df.columns)
    if missing_columns:
        raise AssertionError(f"{target['progress_label']} missing expected columns: {sorted(missing_columns)}")
    assert not any(col.startswith('judge_') for col in out_df.columns), 'Output must not contain judge_* columns'
    assert valid_prediction_mask(out_df).all(), f"{target['progress_label']} output contains invalid classifier prediction rows"
    assert set(out_df['llm_stages_run'].dropna().unique()) == {1}, 'Classifier-only outputs must have llm_stages_run=1'


def assert_has_parsed_rows(out_df: pd.DataFrame, target: dict) -> None:
    parse_success_count = int(out_df['llm_parse_success'].eq(True).sum()) if 'llm_parse_success' in out_df.columns else 0
    if parse_success_count == 0:
        print_parse_diagnostics(out_df, target)
        raise AssertionError(f"{target['progress_label']} produced zero parsed classifier rows")


def print_parse_diagnostics(out_df: pd.DataFrame, target: dict, limit: int = 5) -> None:
    print(f"Parse diagnostics for {target['progress_label']}")
    if out_df.empty:
        print('No output rows available.')
        return
    print(f"Parse successes: {int(out_df['llm_parse_success'].eq(True).sum())}/{len(out_df)}")
    diagnostic_columns = [
        col for col in ['sample_id', 'llm_parse_success', 'llm_raw_response_text', 'llm_evidence']
        if col in out_df.columns
    ]
    print(out_df[diagnostic_columns].head(limit))


def make_failure_result(raw_response_text: str | None = None) -> dict:
    return normalize_classifier_payload({}, raw_response_text, None)


def build_output_row(input_row: pd.Series, result: dict) -> dict:
    label = result['label']
    label_binary = 'benign' if label == 'benign' else 'adversarial'
    row = {'sample_id': input_row['sample_id']}
    for column in GROUND_TRUTH_COLUMNS:
        if column in input_row.index:
            row[column] = input_row[column]
    row.update({
        'llm_pred_binary': label_binary,
        'llm_pred_raw': label,
        'llm_pred_category': result['_category'],
        'llm_conf_binary': result['confidence'],
        'llm_evidence': result.get('evidence', ''),
        'llm_stages_run': 1,
        'llm_provider_name': result['_provider_name'],
        'llm_model_name': result['_model_name'],
        'llm_raw_response_text': result['_raw_response_text'],
        'llm_parse_success': result['_parse_success'],
        'clf_label': label,
        'clf_category': result['_category'],
        'clf_confidence': result['confidence'],
        'clf_evidence': result.get('evidence', ''),
        'clf_nlp_attack_type': result['nlp_attack_type'],
        'clf_provider_name': result['_provider_name'],
        'clf_model_name': result['_model_name'],
        'clf_raw_response_text': result['_raw_response_text'],
        'clf_parse_success': result['_parse_success'],
        'clf_token_logprobs': json.dumps(result.get('_token_logprobs')),
    })
    return row


def write_checkpoint(rows: list[dict], target: dict) -> None:
    if not rows:
        return
    checkpoint = Path(target['checkpoint_path'])
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    new_df = pd.DataFrame(rows)
    if checkpoint.exists():
        existing_df = pd.read_parquet(checkpoint)
        out_df = pd.concat([existing_df, new_df], ignore_index=True)
        out_df = out_df.drop_duplicates(subset=['sample_id'], keep='last')
    else:
        out_df = new_df
    out_df.to_parquet(checkpoint, index=False)
    print(f"Checkpoint rows: {len(out_df):,} -> {checkpoint}")


In [ ]:
from tqdm.auto import tqdm


def batched_rows(df: pd.DataFrame, batch_size: int):
    for start in range(0, len(df), batch_size):
        yield df.iloc[start:start + batch_size]


def run_target(target: dict) -> dict:
    eval_df = prepare_target_df(target)
    checkpoint_path = Path(target['checkpoint_path'])
    output_path = Path(target['output_path'])

    completed_ids = set()
    if checkpoint_path.exists():
        checkpoint_df = pd.read_parquet(checkpoint_path)
        valid_checkpoint_df = checkpoint_df[valid_prediction_mask(checkpoint_df)].copy()
        invalid_checkpoint_rows = len(checkpoint_df) - len(valid_checkpoint_df)
        if invalid_checkpoint_rows:
            print(f"Ignoring invalid checkpoint rows for {target['progress_label']}: {invalid_checkpoint_rows:,}")
        successful_checkpoint_df = valid_checkpoint_df[valid_checkpoint_df['llm_parse_success'].eq(True)].copy()
        completed_ids = set(successful_checkpoint_df['sample_id'].tolist()) if 'sample_id' in successful_checkpoint_df.columns else set()
        print(f"Resuming {target['progress_label']} from checkpoint: {len(completed_ids):,} parsed rows")

    pending_df = eval_df[~eval_df['sample_id'].isin(completed_ids)].reset_index(drop=True)
    print(f"Pending rows for {target['progress_label']}: {len(pending_df):,}")

    buffer: list[dict] = []
    for batch_rows in tqdm(batched_rows(pending_df, BATCH_SIZE), total=(len(pending_df) + BATCH_SIZE - 1) // BATCH_SIZE, desc=f"Classifying {target['name']}"):
        texts = [str(value) for value in batch_rows[text_col].tolist()]
        try:
            results = classify_texts(texts)
        except torch.cuda.OutOfMemoryError:
            raise
        except Exception as exc:
            print(f"Batch failed for {target['progress_label']}; writing parse-failure rows: {exc}")
            results = [make_failure_result(raw_response_text=None) for _ in texts]
        for (_, input_row), result in zip(batch_rows.iterrows(), results):
            buffer.append(build_output_row(input_row, result))
        if len(buffer) >= CHECKPOINT_EVERY:
            write_checkpoint(buffer, target)
            buffer.clear()

    write_checkpoint(buffer, target)

    if checkpoint_path.exists():
        final_df = pd.read_parquet(checkpoint_path)
    else:
        final_df = pd.DataFrame(columns=['sample_id'] + PREDICTION_COLUMNS)

    pre_filter_rows = len(final_df)
    final_df = final_df[valid_prediction_mask(final_df)].copy()
    invalid_final_rows = pre_filter_rows - len(final_df)
    if invalid_final_rows:
        print(f"Dropped invalid final rows for {target['progress_label']}: {invalid_final_rows:,}")
    final_df = final_df.drop_duplicates(subset=['sample_id'], keep='last')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_parquet(output_path, index=False)
    print(f"Final rows: {len(final_df):,} -> {output_path}")

    out_df = pd.read_parquet(output_path)
    assert_valid_output(out_df, target)
    assert_has_parsed_rows(final_df, target)
    assert_has_parsed_rows(out_df, target)
    print_parse_diagnostics(out_df, target, limit=2)
    return {
        'target': target['name'],
        'kind': target['kind'],
        'failure_status': None,
        'rows': len(out_df),
        'parsed_rows': int(out_df['llm_parse_success'].eq(True).sum()),
        'output_path': str(output_path),
    }


TARGETS = [make_main_target(split) for split in MAIN_SPLITS]
TARGETS.extend(make_external_target(dataset_key) for dataset_key in EXTERNAL_DATASETS)

run_results = []
for target in TARGETS:
    print('\n' + '=' * 100)
    print(f"Running {target['progress_label']}")
    run_results.append(run_target(target))

summary_df = pd.DataFrame(run_results)
print('\nBatch output summary')
print(summary_df)


In [ ]:
for result in run_results:
    out_df = pd.read_parquet(result['output_path'])
    print(f"{result['target']}: {len(out_df):,} rows from {result['output_path']}")
    print(out_df['llm_pred_binary'].value_counts(dropna=False))
    print(out_df[['llm_pred_raw', 'llm_conf_binary', 'llm_parse_success']].head())
